# Ejercicio 5 — Comparación $3^3$ vs. Box-Behnken (R)

**Objetivo.** Ajustar el modelo RSM de segundo orden sobre el $3^3$ (27 corridas) y un
Box-Behnken (15 corridas) y comparar coeficientes, errores estándar y punto óptimo.

**Factores:** Temperatura, Tiempo, Concentración
**Respuesta:** Pureza del fármaco (%)

In [ ]:
library(dplyr)
library(ggplot2)
library(rsm)

df3k <- read.csv('../../datos/pureza-farmaco-3k.csv')
cat(sprintf('3^3: %d corridas\n', nrow(df3k)))

## 1. Modelo $3^3$

In [ ]:
formula <- pureza ~ x1+x2+x3+I(x1^2)+I(x2^2)+I(x3^2)+x1:x2+x1:x3+x2:x3
modelo3k <- lm(formula, data=df3k)
print(summary(modelo3k))

## 2. Construir BBD y ajustar modelo

In [ ]:
set.seed(7)
bbd <- bind_rows(
  data.frame(
    x1=c(-1,-1, 1, 1,-1,-1, 1, 1, 0, 0, 0, 0),
    x2=c(-1, 1,-1, 1, 0, 0, 0, 0,-1,-1, 1, 1),
    x3=c( 0, 0, 0, 0,-1, 1,-1, 1,-1, 1,-1, 1)
  ),
  data.frame(x1=c(0,0,0), x2=c(0,0,0), x3=c(0,0,0))
)
# NOTA PEDAGÓGICA: las respuestas del BBD se simulan desde el modelo 3^3 + ruido
# (no son datos independientes). El objetivo es comparar la ESTRUCTURA de diseño
# (varianza de estimación, nº de corridas). En un estudio real ambos diseños serían independientes.
bbd$pureza <- predict(modelo3k, newdata=bbd) + rnorm(nrow(bbd), 0, 0.5)
cat(sprintf('BBD: %d corridas\n', nrow(bbd)))

modelo_bbd <- lm(formula, data=bbd)
print(summary(modelo_bbd))

## 3. Comparación de coeficientes

In [ ]:
terminos <- c('(Intercept)','x1','x2','x3','I(x1^2)','I(x2^2)','I(x3^2)','x1:x2','x1:x3','x2:x3')
nombres  <- c('β₀','β₁','β₂','β₃','β₁₁','β₂₂','β₃₃','β₁₂','β₁₃','β₂₃')
comp <- data.frame(
  Termino = nombres,
  Est_3k  = round(coef(modelo3k)[terminos],3),
  SE_3k   = round(sqrt(diag(vcov(modelo3k)))[terminos],3),
  Est_BBD = round(coef(modelo_bbd)[terminos],3),
  SE_BBD  = round(sqrt(diag(vcov(modelo_bbd)))[terminos],3)
)
print(comp, row.names=FALSE)

## 4. Análisis canónico comparativo

In [ ]:
rsm3k   <- rsm(pureza ~ SO(x1,x2,x3), data=df3k)
rsm_bbd <- rsm(pureza ~ SO(x1,x2,x3), data=bbd)

can3k   <- canonical(rsm3k)
can_bbd <- canonical(rsm_bbd)

centros_r <- c(75,45,3); deltas_r <- c(15,15,1)

cat('Óptimo 3^3 (real):', centros_r + can3k$xs * deltas_r, '\n')
cat('Óptimo BBD (real):', centros_r + can_bbd$xs * deltas_r, '\n')

## 5. Comparación visual de superficies

In [ ]:
options(repr.plot.width=12, repr.plot.height=5)
par(mfrow=c(1,2))
contour(rsm3k,  x1~x2, at=list(x3=can3k$xs['x3']),  image=TRUE,
        col.image=terrain.colors(20), main='3^3 (27 corridas)',
        xlab='x1',ylab='x2')
contour(rsm_bbd, x1~x2, at=list(x3=can_bbd$xs['x3']), image=TRUE,
        col.image=terrain.colors(20), main='BBD (15 corridas)',
        xlab='x1',ylab='x2')

## 6. Conclusión

- El **BBD** estima el mismo modelo de segundo orden con 44% menos corridas
  (el $3^3$ necesita 80% más corridas que el BBD).
- Las superficies son prácticamente idénticas en este ejercicio porque las respuestas del
  BBD se simularon desde el mismo modelo $3^3$ — la comparación ilustra la **estructura**
  de diseño, no diferencias empíricas reales.
- **Regla práctica:** usa $3^k$ cuando necesitas la estructura ortogonal L/Q completa;
  usa BBD/CCD cuando el objetivo es la optimización RSM eficiente.